# PERF-REGRESSION. Paired Benchmark and Regression Thresholds | 配对基准与性能回归阈值

**难度：** Medium　| 　**环境：** CPU 可完成统计与决策练习；单卡 GPU 用于采集配对延迟　| 　**标签：** `性能分析` `Benchmark` `回归阈值`

## 本节导读

单次快慢无法区分候选差异与测量波动。本节用同一试验标识配对 baseline 与 candidate，显式保留失败运行，再用延迟比例的自助区间和质量守护条件作出 `accept / tune / reject`。

## 前置阅读

- [Part 00 · 20 性能证据与实验设计](../00_Prerequisites/20_Performance_Evidence_and_Experiment_Design.ipynb)
- [性能分析 · 02 可重复基线](../topic_discussion/performance_analysis/02_reproducible_baseline.md)

### Step 1：配对条件先于统计方法

配对比较要求 baseline 和 candidate 共享环境、工作负载、试验标识和测量顺序。无法配对的运行应作为失败证据保留，不应静默从分母中删除。

| 字段 | 作用 | 不一致时的处理 | 典型风险 |
| --- | --- | --- | --- |
| `trial_id` | 连接同一次配对试验 | 记录 missing pair | 样本分母改变 |
| `workload_signature` | 固定输入、shape 和模式 | 拒绝合并 | 把 workload 差异当成优化收益 |
| `latency_ms` | 形成 candidate / baseline 比例 | 非正值运行失败 | 除零或错误单位 |
| `failure` | 保留 OOM、超时与执行失败 | 单独计数 | 仅保留成功样本的幸存者偏差 |

![配对运行与失败分母](../docs/public/02_PyTorch_Algorithms/perf_regression_paired_runs.svg)

### Step 2：比例分布比单次差值更稳定

对每个试验计算 `candidate / baseline`，可以消化不同试验的绝对时间尺度。中位数描述中心趋势，自助区间描述在当前样本下的不确定性。

| 统计量 | 含义 | 希望方向 | 使用限制 |
| --- | --- | --- | --- |
| 中位比例 | 典型试验的相对延迟 | 小于 1 表示 candidate 更快 | 不表达不确定性 |
| Bootstrap 区间 | 中位比例的重采样范围 | 整个区间低于阈值 | 依赖配对样本的代表性 |
| 变慢占比 | candidate 比 baseline 慢的试验比例 | 越低越好 | 需与差异幅度同时解读 |

### Step 3：回归阈值是决策契约

回归阈值应在看到 candidate 结果前固定。只有当延迟区间、质量守护条件、失败率和最小样本量同时满足时，才能把候选标记为可采用。

| 条件 | 决策 | 原因 | 后续动作 |
| --- | --- | --- | --- |
| 质量或正确性失败 | `reject` | 性能收益不能覆盖语义退化 | 回滚或修复候选 |
| 有效配对少于门槛 | `tune` | 证据不足 | 增加复测或修复失败运行 |
| 区间整体高于回归阈值 | `reject` | 可观测回归 | 保留失败证据并回退 |
| 区间跨越阈值 | `tune` | 无法区分变化与噪声 | 固定更严格条件后复测 |
| 区间未超阈值且守护条件通过 | `accept` | 无明确回归 | 进入更大 workload 复测 |

![性能回归决策门](../docs/public/02_PyTorch_Algorithms/perf_regression_decision_gate.svg)

### Step 4：实现配对、区间与回归决策

题目使用已经完成的 baseline / candidate 运行记录。你需要建立合法配对，计算延迟比例的重采样区间，再把统计证据与质量、样本量门槛合并为决策。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1：建立配对运行<br>`pair_benchmark_runs` | 按试验标识建立同 workload 配对 | 工作负载签名必须一致；失败和缺失配对单独记录 | 完整配对、缺失配对、签名冲突和失败运行 |
| TODO 2：汇总比例分布<br>`summarize_paired_ratios` | 计算中位比例、变慢占比和 Bootstrap 区间 | 所有延迟为正；随机种子固定；区间单调 | 已知比例、可重复区间与非法延迟 |
| TODO 3：判断性能回归<br>`decide_performance_regression` | 合并质量、样本量和回归阈值 | 质量失败优先拒绝；证据不足或跨阈值时调优 | `accept / tune / reject` 的四类边界 |

In [ ]:
from random import Random
from statistics import median
from typing import Dict, List, Sequence

In [ ]:
def pair_benchmark_runs(
    baseline_runs: Sequence[Dict[str, object]], candidate_runs: Sequence[Dict[str, object]]
) -> Dict[str, object]:
    """Pair successful runs by trial ID while preserving missing and failed runs."""
    baseline = {int(run["trial_id"]): run for run in baseline_runs}
    candidate = {int(run["trial_id"]): run for run in candidate_runs}
    pairs = []
    excluded = {"missing_baseline": [], "missing_candidate": [], "failed": []}
    for trial_id in sorted(set(baseline) | set(candidate)):
        # ==========================================
        # TODO 1：建立配对运行
        # ==========================================
        # missing_baseline = ???  # baseline 中缺少当前 trial。
        # missing_candidate = ???  # candidate 中缺少当前 trial。
        # failed_pair = ???  # 两侧齐全但至少一侧记录失败。
        raise NotImplementedError("TODO 1：请建立配对运行")
        if missing_baseline:
            excluded["missing_baseline"].append(trial_id)
            continue
        if missing_candidate:
            excluded["missing_candidate"].append(trial_id)
            continue
        base, cand = baseline[trial_id], candidate[trial_id]
        if failed_pair:
            excluded["failed"].append(trial_id)
            continue
        if base["workload_signature"] != cand["workload_signature"]:
            raise ValueError("paired runs must share one workload signature")
        pairs.append({"trial_id": trial_id, "baseline_ms": float(base["latency_ms"]),
                      "candidate_ms": float(cand["latency_ms"])})
    return {"pairs": pairs, "excluded": excluded}


def _quantile(values: Sequence[float], probability: float) -> float:
    ordered = sorted(values)
    index = int(round((len(ordered) - 1) * probability))
    return float(ordered[index])


def summarize_paired_ratios(
    pairs: Sequence[Dict[str, float]], bootstrap_samples: int = 1000, seed: int = 42
) -> Dict[str, float]:
    """Summarize candidate/baseline latency ratios with a bootstrap interval."""
    if not pairs or bootstrap_samples <= 0:
        raise ValueError("pairs and bootstrap_samples must be non-empty and positive")
    for pair in pairs:
        if float(pair["baseline_ms"]) <= 0 or float(pair["candidate_ms"]) <= 0:
            raise ValueError("latencies must be positive")
    rng = Random(seed)
    # ==========================================
    # TODO 2：汇总比例分布
    # ==========================================
    # ratios = ???  # 每个合法配对的 candidate / baseline 延迟比例。
    # bootstrap_medians = ???  # 固定随机种子的重采样中位数。
    # ci_low = ???  # 重采样中位数的 5% 分位。
    # ci_high = ???  # 重采样中位数的 95% 分位。
    raise NotImplementedError("TODO 2：请汇总配对比例")
    return {"pair_count": len(ratios), "median_ratio": float(median(ratios)),
            "slower_fraction": sum(ratio > 1.0 for ratio in ratios) / len(ratios),
            "ci_low": ci_low, "ci_high": ci_high}


def decide_performance_regression(
    summary: Dict[str, float], quality_passed: bool, regression_threshold: float, minimum_pairs: int
) -> Dict[str, object]:
    """Apply quality, sample-size, and latency-regression gates in order."""
    if regression_threshold < 0 or minimum_pairs <= 0:
        raise ValueError("threshold must be non-negative and minimum_pairs positive")
    boundary = 1.0 + regression_threshold
    # ==========================================
    # TODO 3：判断性能回归
    # ==========================================
    # quality_failed = ???  # 质量门槛未通过。
    # insufficient_pairs = ???  # 合法配对数不足。
    # confirmed_regression = ???  # 区间下界仍超过回归边界。
    # confirmed_safe = ???  # 区间上界不超过回归边界。
    raise NotImplementedError("TODO 3：请作出性能回归决策")
    if quality_failed:
        decision, reason = "reject", "quality_guard_failed"
    elif insufficient_pairs:
        decision, reason = "tune", "insufficient_pairs"
    elif confirmed_regression:
        decision, reason = "reject", "latency_regression"
    elif confirmed_safe:
        decision, reason = "accept", "no_confirmed_regression"
    else:
        decision, reason = "tune", "interval_crosses_threshold"
    return {"decision": decision, "reason": reason, "regression_boundary": boundary}


### 测试

In [ ]:
def _runs(latencies, signature="shape-a", failed_trial=None):
    """Build deterministic benchmark records for tests."""
    records = []
    for trial_id, latency in enumerate(latencies):
        records.append({
            "trial_id": trial_id,
            "workload_signature": signature,
            "latency_ms": latency,
            "failure": "oom" if trial_id == failed_trial else None,
        })
    return records


def test_pair_benchmark_runs():
    """Validate successful pairs, failures, and missing-trial accounting."""
    baseline = _runs([10.0, 11.0, 9.0])
    candidate = _runs([10.5, 12.0, 9.5], failed_trial=1)[:2]
    report = pair_benchmark_runs(baseline, candidate)
    assert [pair["trial_id"] for pair in report["pairs"]] == [0]
    assert report["excluded"]["failed"] == [1]
    assert report["excluded"]["missing_candidate"] == [2]


def test_summarize_paired_ratios():
    """Validate ratio center, slower fraction, and deterministic interval."""
    pairs = [
        {"trial_id": 0, "baseline_ms": 10.0, "candidate_ms": 9.0},
        {"trial_id": 1, "baseline_ms": 10.0, "candidate_ms": 10.0},
        {"trial_id": 2, "baseline_ms": 10.0, "candidate_ms": 11.0},
    ]
    summary = summarize_paired_ratios(pairs, bootstrap_samples=200, seed=7)
    assert summary["pair_count"] == 3
    assert summary["median_ratio"] == 1.0
    assert summary["slower_fraction"] == 1 / 3
    assert summary["ci_low"] <= summary["median_ratio"] <= summary["ci_high"]


def test_regression_decision():
    """Validate quality, sample-size, clear-regression, and uncertain gates."""
    stable = {"pair_count": 20, "ci_low": 0.98, "ci_high": 1.03, "median_ratio": 1.0}
    regression = {"pair_count": 20, "ci_low": 1.08, "ci_high": 1.12, "median_ratio": 1.1}
    uncertain = {"pair_count": 20, "ci_low": 1.01, "ci_high": 1.08, "median_ratio": 1.04}
    assert decide_performance_regression(stable, True, 0.05, 10)["decision"] == "accept"
    assert decide_performance_regression(regression, True, 0.05, 10)["decision"] == "reject"
    assert decide_performance_regression(uncertain, True, 0.05, 10)["decision"] == "tune"
    assert decide_performance_regression(stable, False, 0.05, 10)["decision"] == "reject"


def run_perf_regression_tests():
    """Run all paired-benchmark and regression-gate tests."""
    test_pair_benchmark_runs()
    test_summarize_paired_ratios()
    test_regression_decision()
    print("✅ 配对基准、Bootstrap 区间与回归门槛测试通过")


run_perf_regression_tests()

---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难，再向下查看参考代码与解析。
<br><br><br><br><br><br><br><br><br><br>

---

## 参考代码与解析

### 代码

In [ ]:
from random import Random
from statistics import median
from typing import Dict, List, Sequence


# TODO 1：建立配对运行
def pair_benchmark_runs(
    baseline_runs: Sequence[Dict[str, object]], candidate_runs: Sequence[Dict[str, object]]
) -> Dict[str, object]:
    """Pair successful runs by trial ID while preserving missing and failed runs."""
    baseline = {int(run["trial_id"]): run for run in baseline_runs}
    candidate = {int(run["trial_id"]): run for run in candidate_runs}
    pairs = []
    excluded = {"missing_baseline": [], "missing_candidate": [], "failed": []}
    for trial_id in sorted(set(baseline) | set(candidate)):
        if trial_id not in baseline:
            excluded["missing_baseline"].append(trial_id)
            continue
        if trial_id not in candidate:
            excluded["missing_candidate"].append(trial_id)
            continue
        base = baseline[trial_id]
        cand = candidate[trial_id]
        if base.get("failure") or cand.get("failure"):
            excluded["failed"].append(trial_id)
            continue
        if base["workload_signature"] != cand["workload_signature"]:
            raise ValueError("paired runs must share one workload signature")
        pairs.append({
            "trial_id": trial_id,
            "baseline_ms": float(base["latency_ms"]),
            "candidate_ms": float(cand["latency_ms"]),
        })
    return {"pairs": pairs, "excluded": excluded}


def _quantile(values: Sequence[float], probability: float) -> float:
    ordered = sorted(values)
    index = int(round((len(ordered) - 1) * probability))
    return float(ordered[index])


# TODO 2：汇总比例分布
def summarize_paired_ratios(
    pairs: Sequence[Dict[str, float]], bootstrap_samples: int = 1000, seed: int = 42
) -> Dict[str, float]:
    """Summarize candidate/baseline latency ratios with a bootstrap interval."""
    if not pairs or bootstrap_samples <= 0:
        raise ValueError("pairs and bootstrap_samples must be non-empty and positive")
    ratios = []
    for pair in pairs:
        baseline_ms = float(pair["baseline_ms"])
        candidate_ms = float(pair["candidate_ms"])
        if baseline_ms <= 0 or candidate_ms <= 0:
            raise ValueError("latencies must be positive")
        ratios.append(candidate_ms / baseline_ms)
    rng = Random(seed)
    bootstrap_medians = [median(rng.choices(ratios, k=len(ratios))) for _ in range(bootstrap_samples)]
    return {
        "pair_count": len(ratios),
        "median_ratio": float(median(ratios)),
        "slower_fraction": sum(ratio > 1.0 for ratio in ratios) / len(ratios),
        "ci_low": _quantile(bootstrap_medians, 0.05),
        "ci_high": _quantile(bootstrap_medians, 0.95),
    }


# TODO 3：判断性能回归
def decide_performance_regression(
    summary: Dict[str, float], quality_passed: bool, regression_threshold: float, minimum_pairs: int
) -> Dict[str, object]:
    """Apply quality, sample-size, and latency-regression gates in order."""
    if regression_threshold < 0 or minimum_pairs <= 0:
        raise ValueError("threshold must be non-negative and minimum_pairs positive")
    boundary = 1.0 + regression_threshold
    if not quality_passed:
        decision, reason = "reject", "quality_guard_failed"
    elif int(summary["pair_count"]) < minimum_pairs:
        decision, reason = "tune", "insufficient_pairs"
    elif float(summary["ci_low"]) > boundary:
        decision, reason = "reject", "latency_regression"
    elif float(summary["ci_high"]) <= boundary:
        decision, reason = "accept", "no_confirmed_regression"
    else:
        decision, reason = "tune", "interval_crosses_threshold"
    return {"decision": decision, "reason": reason, "regression_boundary": boundary}

### 解析

#### TODO 1：建立配对运行

- 只有试验标识和工作负载签名一致的成功运行才能形成延迟配对，缺失与失败仍属于有效性证据。
- 实现分开记录有效配对、缺失、签名冲突和失败；四类测试确认没有选择性丢弃不利样本。

#### TODO 2：汇总比例分布

- 每个配对先形成候选 / 基线比例，再对中位比例重采样，避免绝对时长差异主导结果。
- 实现固定随机种子的 Bootstrap 区间；已知比例、可重复区间和非法延迟测试验证统计口径。

#### TODO 3：判断性能回归

- 正确性和质量先于性能判断；有效配对不足或区间跨越阈值时不能宣称通过。
- 实现按守护条件、样本量和区间依次决策；四类边界测试验证 `accept / tune / reject` 的优先级。

### Step 5：GPU 机制实验——Eager / Compile 配对复测

CPU 练习验证配对和决策规则；GPU 实验在固定输入上交替运行 eager 与 `torch.compile`，记录逐试验延迟、输出一致性和失败。

| 学习问题 | CPU 已确认 | GPU 要确认与后续使用 |
| --- | --- | --- |
| 编译候选的稳态收益是否超过配对噪声且保持正确性 | 配对身份、比例分布、置信区间与回归门槛 | 同一真实层和多个 Token 负载下的首次执行、稳态延迟、输出一致性与失败 |

#### 5.1 环境与固定工作负载

本实验使用 `PERF-CUDA-MECHANISM`。首次运行按[专题 Quickstart](../topic_discussion/performance_analysis/quickstart.md#perf-cuda-mechanism-quickstart)创建并预检环境，页面身份与证据状态见[实验资产表](../topic_discussion/performance_analysis/experiment_assets.md)。

| 配置层 | 本节固定口径 | 修改规则 |
| --- | --- | --- |
| 运行环境 | 单张 CUDA GPU；`PERF-CUDA-MECHANISM` | Profile 预检未返回 `status=ok` 时停止实验 |
| 工作负载 | 同一 TinyMLP、同一权重；1、32、256 三档 Token 负载；Eager 为基线，`torch.compile` 为候选 | 改变负载矩阵、模型或编译选项后使用新运行标签 |
| 配对协议 | 每档单独记录首次调用；随后完成 10 次预热和 30 对交替顺序复测 | 首次调用不混入稳态分布；每对保留原始延迟，失败运行不得静默删除 |
| 决策门槛 | 输出一致；至少 20 对；允许 5% 延迟回归 | 门槛必须在查看候选结果前固定 |
| 结果资产 | `benchmarks/results/perf_regression/<workload_id>_<run_label>.json` | 5.3 从固定相对路径独立读取，不依赖 5.2 内存状态 |

In [3]:
RUN_MODE = "dry_run"  # 默认值 dry_run；改为 real_gpu 后执行真实配对复测。
RUN_LABEL = "default"  # 默认值 default；正式复测使用唯一标签。
ENVIRONMENT_PROFILE = "PERF-CUDA-MECHANISM"  # 预检必须通过的环境标识。
SEMANTIC_ID = "PERF-REGRESSION"  # 机制页稳定标识。
WORKLOAD_ID = "perf-paired-regression"  # 稳定工作负载标识。
WORKLOAD_VERSION = "v4"  # v4 增加多 Token 负载、首次调用成本和逐负载决策。
TOKEN_COUNTS = [1, 32, 256]  # 默认 Token 负载矩阵；同时覆盖低负载噪声区和较高计算负载。
HIDDEN_SIZE = 2048  # 默认隐藏维度；显存不足时下调。
WARMUP_STEPS = 10  # 每条路径的预热次数。
PAIRED_TRIALS = 30  # 默认配对试验数。
REGRESSION_THRESHOLD = 0.05  # 默认允许 5% 延迟回归。
MINIMUM_PAIRS = 20  # 作出 accept/reject 前要求的最少合法配对数。
BOOTSTRAP_SAMPLES = 2000  # 自助重采样次数；增大后区间更稳定但统计时间更长。
SEED = 42  # 固定输入和自助重采样的随机种子。
RESULT_RELATIVE_DIR = "benchmarks/results/perf_regression"  # 结果 JSON 的仓库相对目录。

#### 5.2 执行配对复测并保存 JSON

In [4]:
from pathlib import Path
import json
import sys
from random import Random
from statistics import median
from time import perf_counter
import torch


def _gpu_quantile(values, probability):
    ordered = sorted(values)
    index = int(round((len(ordered) - 1) * probability))
    return float(ordered[index])


def _gpu_summarize_paired_ratios(pairs, bootstrap_samples, seed):
    """汇总 Step5 真实配对延迟，不依赖 Step4 参考答案。"""
    ratios = [float(pair["candidate_ms"]) / float(pair["baseline_ms"]) for pair in pairs]
    if not ratios or any(not (ratio > 0) for ratio in ratios):
        raise ValueError("配对延迟必须为正且不能为空")
    rng = Random(seed)
    bootstrap_medians = [median(rng.choices(ratios, k=len(ratios))) for _ in range(bootstrap_samples)]
    return {
        "pair_count": len(ratios),
        "median_ratio": float(median(ratios)),
        "slower_fraction": sum(ratio > 1.0 for ratio in ratios) / len(ratios),
        "ci_low": _gpu_quantile(bootstrap_medians, 0.05),
        "ci_high": _gpu_quantile(bootstrap_medians, 0.95),
    }


def _gpu_decide_regression(summary, quality_passed, threshold, minimum_pairs):
    boundary = 1.0 + threshold
    if not quality_passed:
        decision, reason = "reject", "quality_guard_failed"
    elif summary["pair_count"] < minimum_pairs:
        decision, reason = "tune", "insufficient_pairs"
    elif summary["ci_low"] > boundary:
        decision, reason = "reject", "latency_regression"
    elif summary["ci_high"] <= boundary:
        decision, reason = "accept", "no_confirmed_regression"
    else:
        decision, reason = "tune", "interval_crosses_threshold"
    return {"decision": decision, "reason": reason, "regression_boundary": boundary}

if RUN_MODE == "real_gpu":
    if not torch.cuda.is_available():
        raise RuntimeError("PERF-REGRESSION 真实实验需要 CUDA GPU")
    from tools.project_runtime import resolve_project_root

    class TinyMLP(torch.nn.Module):
        def __init__(self, hidden_size):
            super().__init__()
            self.up = torch.nn.Linear(hidden_size, hidden_size * 2, bias=False)
            self.down = torch.nn.Linear(hidden_size * 2, hidden_size, bias=False)

        def forward(self, inputs):
            return self.down(torch.nn.functional.silu(self.up(inputs)))

    def measure_steady(function, inputs):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            output = function(inputs)
        end.record()
        end.synchronize()
        return float(start.elapsed_time(end)), output

    def measure_first_execution(function, inputs):
        torch.cuda.synchronize()
        started_at = perf_counter()
        with torch.no_grad():
            output = function(inputs)
        torch.cuda.synchronize()
        return (perf_counter() - started_at) * 1000.0, output

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    device = torch.device("cuda")
    eager = TinyMLP(HIDDEN_SIZE).to(device).eval()
    load_results = []
    for token_count in TOKEN_COUNTS:
        inputs = torch.randn(token_count, HIDDEN_SIZE, device=device)
        pairs = []
        load_failure = None
        quality_passed = False
        first_execution = None
        try:
            candidate = torch.compile(eager, dynamic=False)
            eager_first_ms, eager_first_output = measure_first_execution(eager, inputs)
            candidate_first_ms, candidate_first_output = measure_first_execution(candidate, inputs)
            first_execution = {
                "eager_ms": eager_first_ms,
                "compile_ms": candidate_first_ms,
                "compile_to_eager_ratio": candidate_first_ms / eager_first_ms,
            }
            for _ in range(WARMUP_STEPS):
                measure_steady(eager, inputs)
                measure_steady(candidate, inputs)
            for trial_id in range(PAIRED_TRIALS):
                first, second = (eager, candidate) if trial_id % 2 == 0 else (candidate, eager)
                first_ms, _ = measure_steady(first, inputs)
                second_ms, _ = measure_steady(second, inputs)
                eager_ms, candidate_ms = (first_ms, second_ms) if trial_id % 2 == 0 else (second_ms, first_ms)
                pairs.append({"trial_id": trial_id, "baseline_ms": eager_ms, "candidate_ms": candidate_ms})
            quality_passed = bool(torch.allclose(eager_first_output, candidate_first_output, atol=1e-3, rtol=1e-3))
        except Exception as error:
            load_failure = {"type": type(error).__name__, "message": str(error)}
            pairs = []
        summary = _gpu_summarize_paired_ratios(pairs, BOOTSTRAP_SAMPLES, SEED + token_count) if pairs else None
        load_decision = (
            _gpu_decide_regression(summary, quality_passed, REGRESSION_THRESHOLD, MINIMUM_PAIRS)
            if summary else {"decision": "reject", "reason": "candidate_execution_failed"}
        )
        load_results.append({
            "token_count": token_count,
            "first_execution": first_execution,
            "quality": {"outputs_allclose": quality_passed},
            "summary": summary,
            "paired_runs": pairs,
            "failure": load_failure,
            "decision": load_decision,
        })

    result_dir = resolve_project_root() / RESULT_RELATIVE_DIR
    result_dir.mkdir(parents=True, exist_ok=True)
    result_path = result_dir / f"{WORKLOAD_ID}_{RUN_LABEL}.json"
    decisions = [item["decision"]["decision"] for item in load_results]
    if any(decision == "reject" for decision in decisions):
        overall_decision = {"decision": "reject", "reason": "one_or_more_loads_rejected"}
    elif any(decision == "tune" for decision in decisions):
        overall_decision = {"decision": "tune", "reason": "one_or_more_loads_inconclusive"}
    else:
        overall_decision = {"decision": "accept", "reason": "no_confirmed_regression_across_loads"}
    failures = [{"token_count": item["token_count"], **item["failure"]} for item in load_results if item["failure"]]
    record = {
        "schema_version": "performance-regression-mechanism/v4",
        "profile_id": ENVIRONMENT_PROFILE,
        "semantic_id": SEMANTIC_ID,
        "run_label": RUN_LABEL,
        "framework": {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda},
        "hardware": {"device": torch.cuda.get_device_name(0), "total_memory_bytes": torch.cuda.get_device_properties(0).total_memory},
        "workload": {
            "id": WORKLOAD_ID,
            "version": WORKLOAD_VERSION,
            "token_counts": TOKEN_COUNTS,
            "hidden_size": HIDDEN_SIZE,
            "warmup_steps": WARMUP_STEPS,
            "paired_trials": PAIRED_TRIALS,
            "minimum_pairs": MINIMUM_PAIRS,
            "bootstrap_samples": BOOTSTRAP_SAMPLES,
            "seed": SEED,
        },
        "metrics": {"load_matrix": load_results},
        "failure": failures or None,
        "evidence_level": "local_mechanism_benchmark",
        "decision": overall_decision,
    }
    result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    print("⏭️ RUN_MODE=dry_run；未执行 GPU 配对复测。")

#### 5.3 读取结果

In [5]:
from pathlib import Path
import json
from tools.project_runtime import resolve_project_root

if RUN_MODE == "real_gpu":
    result_path = resolve_project_root() / RESULT_RELATIVE_DIR / f"{WORKLOAD_ID}_{RUN_LABEL}.json"
    if not result_path.exists():
        raise FileNotFoundError(f"未找到结果：{result_path}")
    regression_record = json.loads(result_path.read_text(encoding="utf-8"))
    print(json.dumps(regression_record, ensure_ascii=False, indent=2))
else:
    regression_record = {"status": "not_run", "decision": "enable_real_gpu_after_profile_preflight"}
    print(regression_record)

{
  "schema_version": "performance-regression-mechanism/v3",
  "profile_id": "PERF-CUDA-MECHANISM",
  "semantic_id": "PERF-REGRESSION",
  "run_label": "rtx5070ti_regression_v2",
  "framework": {
    "python": "3.12.15",
    "torch": "2.11.0+cu128",
    "cuda": "12.8"
  },
  "hardware": {
    "device": "NVIDIA GeForce RTX 5070 Ti Laptop GPU",
    "total_memory_bytes": 12328960000
  },
  "workload": {
    "id": "perf-paired-regression",
    "version": "v3",
    "batch_size": 32,
    "hidden_size": 2048,
    "warmup_steps": 10,
    "paired_trials": 30,
    "minimum_pairs": 20,
    "bootstrap_samples": 2000,
    "seed": 42
  },
  "quality": {
    "outputs_allclose": true
  },
  "metrics": {
    "summary": {
      "pair_count": 30,
      "median_ratio": 0.9983113158047896,
      "slower_fraction": 0.43333333333333335,
      "ci_low": 0.992499925020227,
      "ci_high": 1.0008142279397152
    },
    "paired_runs": [
      {
        "trial_id": 0,
        "baseline_ms": 0.22150400280952454,
 

#### 5.4 解释配对结果

RTX 5070 Ti 实测中，三档负载均通过输出一致性检查且没有执行失败。编译候选没有在任一负载下获得稳态加速；随着 Token 数增加，延迟差距由明确回归逐步缩小。

| 观察项 | 读取字段 | 当前结果 | 解释与动作 |
| --- | --- | --- | --- |
| 正确性与执行 | `failure`、`quality.outputs_allclose` | 三档均 `allclose=true`；无失败 | 当前差异可以进入性能判断，不由正确性失败解释 |
| 首次调用代价 | `first_execution.eager_ms`、`compile_ms` 与比例 | 1 Token：7.27 / 404.44 ms；32 Token：0.46 / 29.00 ms；256 Token：1.11 / 245.67 ms | 候选首次调用包含编译等待且受 shape 与缓存影响，不混入稳态分布 |
| 稳态回归 | `summary.median_ratio` 与 90% Bootstrap 区间 | 1 Token：1.089，区间 [1.082, 1.100]；32 Token：1.055，[1.050, 1.063]；256 Token：1.017，[1.014, 1.020] | 1 Token 确认超过 5% 回归；32 Token 跨越门槛；256 Token 未越过门槛但仍略慢 |
| 适用边界 | 三档逐负载决策与总决策 | `reject / tune / accept`；总决策 `reject` | `accept` 仅表示未确认超过回归门槛；三档均未证明编译加速，不能将 256 Token 的 `accept` 表述为性能收益 |

## 相关阅读

- [性能分析 · Task5 回归、噪声与阈值](../topic_discussion/performance_analysis/intro.md)
- [Part 02 · 66 推理性能比较](./66_Inference_Performance_Comparison.ipynb)
- [Part 02 · 73 训练性能分析](./73_Training_Performance_Analysis.ipynb)